In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler,PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [3]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [4]:
train_sample = pd.read_csv(train_sample_path)
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              25599 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                25622 non-null  str    
 8   population_density             25552 non-null  str    
 9   weather                        25571 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [5]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [6]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              25599 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                25622 non-null  str    
 8   population_density             25552 non-null  str    
 9   weather                        25571 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [7]:
train_sample.isnull().sum()

start_point                          0
end_point                            0
time_of_day                          0
day_of_week                          0
traffic_condition                14401
event_count                          0
is_holiday                           0
vehicle_density                  14378
population_density               14448
weather                          14429
public_transport_availability        0
historical_delay_factor              0
travel_time                          0
dtype: int64

In [8]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'start_end_point',
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = train_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            train_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            train_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        train_sample.loc[mask, target] = filled.values

In [9]:
train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
dtype: int64

In [10]:
test_sample.isnull().sum()

start_point                        0
end_point                          0
time_of_day                        0
day_of_week                        0
traffic_condition                600
event_count                        0
is_holiday                         0
vehicle_density                  600
population_density               600
weather                          600
public_transport_availability      0
historical_delay_factor            0
dtype: int64

In [11]:
reference_cols = [
    'start_point',
    'end_point',
    'time_of_day',
    'day_of_week',
    'event_count',
    'is_holiday',
    'public_transport_availability',
    'start_end_point'
]

target_cols = [
    'traffic_condition',
    'vehicle_density',
    'population_density',
    'weather'
]

# Urutan level fallback
reference_levels = [
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'event_count',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week',
        'is_holiday',
        'public_transport_availability'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day',
        'day_of_week'
    ],
    [
        'start_point',
        'end_point',
        'time_of_day'
    ],
    [
        'start_point',
        'end_point'
    ]
]

for target in target_cols:

    for refs in reference_levels:

        # Hanya proses yang masih missing
        mask = test_sample[target].isna()

        if not mask.any():
            break

        # Cari modus berdasarkan grup
        mode_map = (
            test_sample.dropna(subset=[target])
              .groupby(refs)[target]
              .agg(lambda x: x.mode().iloc[0])
        )

        # Mapping ke baris yang masih missing
        filled = (
            test_sample.loc[mask, refs]
              .merge(
                  mode_map.rename('mode_value'),
                  left_on=refs,
                  right_index=True,
                  how='left'
              )['mode_value']
        )

        # Isi hanya yang berhasil mendapatkan modus
        test_sample.loc[mask, target] = filled.values

In [12]:
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
weather                          0
public_transport_availability    0
historical_delay_factor          0
dtype: int64

In [13]:
train_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    40000 non-null  str    
 1   end_point                      40000 non-null  str    
 2   time_of_day                    40000 non-null  str    
 3   day_of_week                    40000 non-null  str    
 4   traffic_condition              40000 non-null  float64
 5   event_count                    40000 non-null  int64  
 6   is_holiday                     40000 non-null  int64  
 7   vehicle_density                40000 non-null  str    
 8   population_density             40000 non-null  str    
 9   weather                        40000 non-null  str    
 10  public_transport_availability  40000 non-null  int64  
 11  historical_delay_factor        40000 non-null  float64
 12  travel_time                    40000 non-null  float64
dt

In [14]:
test_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    3000 non-null   str    
 1   end_point                      3000 non-null   str    
 2   time_of_day                    3000 non-null   str    
 3   day_of_week                    3000 non-null   str    
 4   traffic_condition              3000 non-null   float64
 5   event_count                    3000 non-null   int64  
 6   is_holiday                     3000 non-null   int64  
 7   vehicle_density                3000 non-null   str    
 8   population_density             3000 non-null   str    
 9   weather                        3000 non-null   str    
 10  public_transport_availability  3000 non-null   int64  
 11  historical_delay_factor        3000 non-null   float64
dtypes: float64(2), int64(3), str(7)
memory usage: 281.4 KB


In [15]:
train_sample[
    (train_sample["traffic_condition"] >= 9) &
    (train_sample["vehicle_density"].isin(["low", "medium"]))
]["vehicle_density"].value_counts()

vehicle_density
medium    16295
low       10253
Name: count, dtype: int64

In [16]:
condition = (
    (train_sample["traffic_condition"] >= 9) &
    (train_sample["vehicle_density"].isin(["low", "medium"]))
)

train_sample.loc[condition, "vehicle_density"] = "high"

train_sample[
    (train_sample["traffic_condition"] >= 9) &
    (train_sample["vehicle_density"].isin(["low", "medium"]))
]["vehicle_density"].value_counts()

Series([], Name: count, dtype: int64)

In [27]:
travel_time_outlier = train_sample[train_sample['travel_time'] > 150]

In [29]:
travel_time_outlier['travel_time']

4621     151.533099
12796    154.854552
15135    153.297186
16343    205.679832
18513    218.832465
21716    168.697931
23387    153.943082
24718    160.187804
26363    174.899877
28747    179.937114
Name: travel_time, dtype: float64

In [30]:
travel_time_outlier = train_sample[
    train_sample['travel_time'] > 150
]

condition = (
    (train_sample['traffic_condition'] >= 9)
    & (train_sample['travel_time'] < 30)
    & (train_sample['weather'].isin(["storm", "rain"]))
    & (train_sample['start_point'] == 'North Jakarta (Jakarta Utara)')
    & (train_sample['time_of_day'] != 'night')
)

train_sample.loc[condition, 'travel_time'] = (
    travel_time_outlier['travel_time'].mean()
)

In [32]:
time = train_sample["time_of_day"].str.split().str[0]
day = train_sample["day_of_week"].str.split().str[0]

train_sample["day_and_time"] = day + " " + time
test_sample["day_and_time"] = test_sample["day_of_week"].str.split().str[0] + " " + test_sample["time_of_day"].str.split().str[0]

In [33]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["route"] = start + " " + end
test_sample["route"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [34]:
route = train_sample["route"]
day_and_time = train_sample["day_and_time"]

train_sample["route_day_and_time"] = route + " " + day_and_time
test_sample["route_day_and_time"] = test_sample["route"] + " " + test_sample["day_and_time"]

In [35]:
route = train_sample["route"]

train_sample["route_public_transport"] = route + " " + train_sample["public_transport_availability"].astype(str)
test_sample["route_public_transport"] = test_sample["route"] + " " + test_sample["public_transport_availability"].astype(str)

In [36]:
train_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time,day_and_time,route,route_day_and_time,route_public_transport
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,9.0,9,1,high,high,rain,1,0.878909,26.907612,Sunday day,West South,West South Sunday day,West South 1
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,9.0,7,1,high,high,fog,1,1.081668,27.489129,Thursday morning,West South,West South Thursday morning,West South 1


In [37]:
X_train = train_sample.drop(columns=['travel_time'])
y_train = train_sample['travel_time']

In [38]:
X_test = test_sample

In [39]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_train index:", X_train.index[:5])
print("y_train index:", y_train.index[:5])

X_train: (40000, 16)
y_train: (40000,)
X_train index: RangeIndex(start=0, stop=5, step=1)
y_train index: RangeIndex(start=0, stop=5, step=1)


one-hot encoding

In [40]:
train_sample.nunique().sort_values(ascending=False)

historical_delay_factor          40000
travel_time                      38816
route_day_and_time                 280
day_and_time                        28
route_public_transport              10
route                               10
event_count                          8
day_of_week                          7
traffic_condition                    6
start_point                          4
time_of_day                          4
weather                              4
end_point                            4
vehicle_density                      3
population_density                   3
public_transport_availability        3
is_holiday                           2
dtype: int64

In [41]:
categorical_cols = train_sample.select_dtypes(include=['str', 'object']).columns
numeric_cols = train_sample.select_dtypes(include='number').columns.drop('travel_time', errors='ignore')

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

R²: 0.4972586046832442
MAE: 9.8642441144598
MSE: 400.8336085393579


In [43]:
model.fit(X_train, y_train)

y_pred = model.predict(test_sample)

pd.DataFrame(y_pred).to_csv('submission.csv', index=False)

In [44]:
pd.read_csv('submission.csv')['0']

0       48.595695
1        8.584760
2       28.248008
3       17.946758
4       40.527794
          ...    
2995    35.709022
2996    17.393923
2997     7.333094
2998    76.491426
2999    12.243441
Name: 0, Length: 3000, dtype: float64